# Stage 6 : Generative / foundation models : TerraMind, Gemini and GPT

**Purpose:** test whether modern gGen AI models add value compared with our own ViT
**Rules:** prompts and all settings are chosen on **validation** data only.

## 0. Install TerraMind's library

In [ ]:
!pip -q install "terratorch>=1.2.5" "numpy<2.3" "pandas==2.2.3" "python-dateutil<3" timm google-genai openai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip -q install "pandas==2.2.3" "python-dateutil<3" "numpy<2.3"

In [ ]:
import sys, os, json, time, random, hashlib, zipfile, io, math, base64, re, warnings

## 1. Setup, split, metrics and log

In [ ]:

from pathlib import Path
import numpy as np, pandas as pd

SEED = 42; FINAL_SEEDS = [42, 43, 44]
QUICK = os.environ.get("FN_QUICK") == "1"               # offline testing only — leave unset
IN_COLAB = "google.colab" in sys.modules
if os.environ.get("FORESTNET_BASE"):
    BASE = Path(os.environ["FORESTNET_BASE"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/forestnet_project")
else:
    BASE = Path("./forestnet_project")
RUNS6 = BASE / "outputs" / "stage6" / "runs"; RUNS6.mkdir(parents=True, exist_ok=True)
OUT6 = BASE / "outputs" / "stage6"; CACHE = BASE / "cache"
LOG_PATH = BASE / "experiment_log.csv"

import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score, recall_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)
DEV = "cuda" if torch.cuda.is_available() else "cpu"; AMP = DEV == "cuda"
print("torch", torch.__version__, "| device", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

meta = json.load(open(BASE / "outputs" / "dataset_meta.json"))
assert sha256(BASE / "outputs" / "split_manifest.csv") == meta["manifest_sha256"], "Split changed!"
man = pd.read_csv(BASE / "outputs" / "split_manifest.csv").sort_values("example_path").reset_index(drop=True)
CLASSES = sorted(man["merged_label"].unique()); CLS2ID = {c: i for i, c in enumerate(CLASSES)}
man["y"] = man["merged_label"].map(CLS2ID)
P_ID, S_ID, O_ID, G_ID = (CLS2ID["Plantation"], CLS2ID["Smallholder agriculture"], CLS2ID["Other"], CLS2ID["Grassland shrubland"])
TR_IDX = np.where(man.split == "train")[0]; VA_IDX = np.where(man.split == "val")[0]

def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    rec = recall_score(y_true, y_pred, labels=range(len(CLASSES)), average=None, zero_division=0)
    return {"macro_f1": f1_score(y_true, y_pred, labels=range(len(CLASSES)), average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred), "recall_plantation": rec[P_ID], "recall_other": rec[O_ID],
            "recall_smallholder": rec[S_ID], "recall_grassland": rec[G_ID],
            "rate_P_to_S": float((y_pred[y_true == P_ID] == S_ID).mean()) if (y_true == P_ID).any() else np.nan,
            "rate_O_to_SG": float(np.isin(y_pred[y_true == O_ID], [S_ID, G_ID]).mean()) if (y_true == O_ID).any() else np.nan}

LOG_COLS = ["run_id", "date", "stage", "model", "change", "hyperparameters", "seed", "val_macro_f1",
            "val_recall_plantation", "val_recall_other", "train_time_s", "infer_ms_per_img", "outcome", "note"]
def log_run(run_id, model, change, params, seed, m=None, train_s=None, infer_ms=None, outcome="ok", note=""):
    m = m or {}
    row = {"run_id": run_id, "date": time.strftime("%Y-%m-%d %H:%M"), "stage": "6", "model": model, "change": change,
           "hyperparameters": json.dumps(params, default=str), "seed": seed,
           "val_macro_f1": round(m.get("macro_f1", np.nan), 4), "val_recall_plantation": round(m.get("recall_plantation", np.nan), 4),
           "val_recall_other": round(m.get("recall_other", np.nan), 4), "train_time_s": None if train_s is None else round(train_s, 1),
           "infer_ms_per_img": None if infer_ms is None else round(infer_ms, 2), "outcome": outcome, "note": note}
    pd.DataFrame([row], columns=LOG_COLS).to_csv(LOG_PATH, mode="a", header=not LOG_PATH.exists(), index=False)

def add_lesson(run_id, text):
    log = pd.read_csv(LOG_PATH); hit = log.run_id == run_id
    assert hit.any(), f"{run_id} not in log"
    log.loc[hit, "note"] = log.loc[hit, "note"].fillna("").astype(str) + " | Lesson: " + text
    log.to_csv(LOG_PATH, index=False); print("Lesson added to", run_id)

Mounted at /content/drive
torch 2.11.0+cu130 | device cuda Tesla T4


## Part A : TerraMind-1.0-base, fine-tuned (RGB)


In [ ]:
COMP = torch.from_numpy(np.load(CACHE / "comp_full.npy"))
MEAN = torch.tensor(meta["train_rgb_norm"]["mean"]).view(1, 3, 1, 1)
STD = torch.tensor(meta["train_rgb_norm"]["std"]).view(1, 3, 1, 1)
Y = torch.tensor(man.y.values)
COMP_GPU = COMP.to(DEV); Y_GPU = Y.to(DEV); MEAN_G, STD_G = MEAN.to(DEV), STD.to(DEV)

def make_batch(idx, train=False):
    x = (COMP_GPU[idx].permute(0, 3, 1, 2).float() / 255.0 - MEAN_G) / STD_G
    if train:
        B = x.shape[0]
        x = torch.where((torch.rand(B, device=DEV) < 0.5).view(B, 1, 1, 1), x.flip(3), x)
        x = torch.where((torch.rand(B, device=DEV) < 0.5).view(B, 1, 1, 1), x.flip(2), x)
        k = torch.randint(0, 4, (B,), device=DEV); out = x.clone()
        for r in (1, 2, 3):
            m = k == r
            if m.any():
                out[m] = torch.rot90(x[m], r, dims=(2, 3))
        x = out
    return x

class TerraMindClassifier(nn.Module):
    def __init__(self, pretrained=True, drop=0.2):
        super().__init__()
        from terratorch.registry import BACKBONE_REGISTRY
        self.backbone = BACKBONE_REGISTRY.build("terramind_v1_base", pretrained=pretrained, modalities=["RGB"])
        self.head = nn.Sequential(nn.LayerNorm(768), nn.Dropout(drop), nn.Linear(768, len(CLASSES)))
    def forward(self, x):
        return self.head(self.backbone(x)[-1].mean(1))

counts = np.bincount(man.y.values[TR_IDX], minlength=len(CLASSES))
CLASS_W = torch.tensor(len(TR_IDX) / (len(CLASSES) * counts), dtype=torch.float32, device=DEV)
MAX_EPOCHS, PATIENCE, BS = (2, 1, 8) if QUICK else (30, 7, 32)
PRETRAINED = os.environ.get("FN_PRETRAINED", "1") == "1"

@torch.no_grad()
def predict_idx(model, idx_np, bs=64):
    model.eval(); out = []
    t = torch.tensor(idx_np, device=DEV)
    for b in range(0, len(t), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            out.append(model(make_batch(t[b:b + bs])).float().cpu())
    return torch.cat(out)

def run_terramind(run_id, lr, seed, wd=0.01, save=False):
    rdir = RUNS6 / run_id
    if (rdir / "result.json").exists():
        r = json.load(open(rdir / "result.json")); print(run_id, "already done:", round(r["val"]["macro_f1"], 4)); return r
    rdir.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    params = {"lr": lr, "wd": wd, "bs": BS, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "input": "RGB 224, train-stat standardised"}
    try:
        model = TerraMindClassifier(PRETRAINED).to(DEV)
    except Exception as e:
        log_run(run_id, "TerraMind-1.0-base", "fine-tune", params, seed, outcome="failed (model load)", note=repr(e)[:200])
        print(run_id, "FAILED to build TerraMind:", repr(e)); return None
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = math.ceil(len(TR_IDX) / BS); total, warm = steps * MAX_EPOCHS, steps
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    crit = nn.CrossEntropyLoss(weight=CLASS_W); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    tr_t = torch.tensor(TR_IDX, device=DEV); hist, best, best_ep, best_state, bad, outcome = [], -1, 0, None, 0, "ok"
    t0 = time.time()
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); perm = tr_t[torch.randperm(len(tr_t), device=DEV)]; run_loss = 0.0; nan = False
        for b in range(steps):
            idx = perm[b * BS:(b + 1) * BS]
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                loss = crit(model(make_batch(idx, train=True)).float(), Y_GPU[idx])
            if not torch.isfinite(loss):
                nan = True; break
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item() * len(idx)
        if nan:
            outcome = "failed (diverged)"; break
        va_logits = predict_idx(model, VA_IDX); va_f1 = f1_score(man.y.values[VA_IDX], va_logits.argmax(1).numpy(), average="macro")
        va_loss = F.cross_entropy(va_logits, Y[VA_IDX], weight=CLASS_W.cpu()).item()
        hist.append({"epoch": ep, "train_loss_aug": run_loss / len(TR_IDX), "val_loss": va_loss, "val_macro_f1": va_f1, "time_s": time.time() - t0})
        print(f"  {run_id} epoch {ep}: val macro-F1 {va_f1:.4f}")
        if va_f1 > best:
            best, best_ep, bad = va_f1, ep, 0; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if bad >= PATIENCE:
            break
    train_s = time.time() - t0; pd.DataFrame(hist).to_csv(rdir / "history.csv", index=False)
    if best_state is None:
        log_run(run_id, "TerraMind-1.0-base", "fine-tune", params, seed, train_s=train_s, outcome=outcome); return None
    model.load_state_dict(best_state)
    va_pred = predict_idx(model, VA_IDX).argmax(1).numpy(); m = evaluate(man.y.values[VA_IDX], va_pred)
    np.save(rdir / "val_pred.npy", va_pred)
    if save:
        torch.save(best_state, rdir / "best.pt")
    res = {"run_id": run_id, "lr": lr, "seed": seed, "val": m, "best_epoch": best_ep, "epochs_run": len(hist), "train_s": train_s,
           "params_M": sum(p.numel() for p in model.parameters()) / 1e6, "outcome": outcome}
    json.dump(res, open(rdir / "result.json", "w"), indent=2, default=str)
    log_run(run_id, "TerraMind-1.0-base", "fine-tune" + (" (final seed)" if save else " (lr search)"), params, seed, m, train_s,
            note=f"best epoch {best_ep}/{len(hist)}")
    print(f"{run_id}: val macro-F1 {m['macro_f1']:.4f} | best epoch {best_ep}/{len(hist)} | {train_s / 60:.1f} min")
    del model, opt, sched, scaler, best_state           # free GPU memory before the next run
    import gc; gc.collect(); torch.cuda.empty_cache() if DEV == "cuda" else None
    return res

In [ ]:
LRS = [1e-5, 3e-5, 1e-4] if not QUICK else [1e-4]
tm_search = [run_terramind(f"S6-TM-LR{lr:.0e}", lr, SEED, save=True) for lr in LRS]
tm_search = [r for r in tm_search if r]
if tm_search:
    best_lr = max(tm_search, key=lambda r: r["val"]["macro_f1"])["lr"]
    print("Best TerraMind learning rate:", best_lr)
    for r in tm_search:                                   # keep weights only for the chosen learning rate (saves Drive space)
        if r["lr"] != best_lr and (RUNS6 / r["run_id"] / "best.pt").exists():
            (RUNS6 / r["run_id"] / "best.pt").unlink()

S6-TM-LR1e-05 already done: 0.73
S6-TM-LR3e-05 already done: 0.7365
S6-TM-LR1e-04 already done: 0.7314
Best TerraMind learning rate: 3e-05


In [ ]:
if tm_search:
    for s in FINAL_SEEDS[1:]:
        run_terramind(f"S6-TM-FINAL-s{s}", best_lr, s, save=True)
    best_42 = f"S6-TM-LR{best_lr:.0e}"
    tm_final = [json.load(open(RUNS6 / rid / "result.json")) for rid in [best_42] + [f"S6-TM-FINAL-s{s}" for s in FINAL_SEEDS[1:]]
                if (RUNS6 / rid / "result.json").exists()]
    tm_df = pd.DataFrame([{**r["val"], "seed": r["seed"], "train_min": r["train_s"] / 60} for r in tm_final])
    display(tm_df.round(3))
    json.dump({"best_lr": best_lr, "final_run_ids": [best_42] + [f"S6-TM-FINAL-s{s}" for s in FINAL_SEEDS[1:]]},
              open(OUT6 / "terramind_final.json", "w"), indent=2)

S6-TM-FINAL-s43 already done: 0.7664
S6-TM-FINAL-s44 already done: 0.727


,macro_f1,accuracy,recall_plantation,recall_other,recall_smallholder,recall_grassland,rate_P_to_S,rate_O_to_SG,seed,train_min
0,0.737,0.770,0.784,0.814,0.743,0.711,0.142,0.143,42,3.157
1,0.766,0.814,0.908,0.700,0.807,0.556,0.069,0.171,43,8.328
2,0.727,0.774,0.817,0.743,0.793,0.556,0.101,0.214,44,2.975


In [ ]:
# TerraMind GPU latency per image (one image at a time, from cache)
if tm_search:
    m = TerraMindClassifier(pretrained=False).to(DEV)
    m.load_state_dict(torch.load(RUNS6 / best_42 / "best.pt", map_location="cpu")); m.eval()
    ts = []
    for i in VA_IDX[:30]:
        if DEV == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            m(make_batch(torch.tensor([i], device=DEV)))
        if DEV == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    TM_LAT = float(np.median(ts[5:])); print(f"TerraMind latency: {TM_LAT:.1f} ms per image on {DEV}")
    del m; torch.cuda.empty_cache() if DEV == "cuda" else None

TerraMind latency: 15.5 ms per image on cuda


## Parts B and C :Vision-language models (Gemini, GPT): zero-shot and few-shot


In [ ]:
from getpass import getpass

def get_key(name):
    try:
        from google.colab import userdata
        k = userdata.get(name)
        if k: return k.strip()
    except Exception:
        pass
    k = getpass(f"Paste your {name} (or just press Enter to skip): ").strip()
    return k or None

GEMINI_KEY = get_key("GEMINI_API_KEY")
OPENAI_KEY = get_key("OPENAI_API_KEY")
print("Gemini key found:", bool(GEMINI_KEY), "| OpenAI key found:", bool(OPENAI_KEY))

if GEMINI_KEY:
    from google import genai
    from google.genai import types
    gclient = genai.Client(api_key=GEMINI_KEY)
    print("Gemini models (pick one with 'flash'):")
    print([m.name.replace("models/", "") for m in gclient.models.list() if "gemini" in m.name][:40])
if OPENAI_KEY:
    from openai import OpenAI
    oclient = OpenAI(api_key=OPENAI_KEY)
    print("OpenAI models:")
    print(sorted(m.id for m in oclient.models.list() if m.id.startswith(("gpt", "o")))[:60])

Paste your GEMINI_API_KEY (or just press Enter to skip): ··········
Paste your OPENAI_API_KEY (or just press Enter to skip): ··········
Gemini key found: True | OpenAI key found: True
Gemini models (pick one with 'flash'):
['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-2.5-flash-preview-tts', 'gemini-2.5-pro-preview-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-pro-latest', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-image', 'gemini-3-flash-preview', 'gemini-3.1-pro-preview', 'gemini-3.1-pro-preview-customtools', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-lite', 'gemini-3-pro-image-preview', 'gemini-3-pro-image', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-lite-image', 'gemini-nano-banana-2.1', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-omni-flash-preview', 'gemini-omni-1.1-flash', 'gemini-3.5-transcribe', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-4-argon', 'gemini-3.1-flash-tts-preview', 'g

In [ ]:
if GEMINI_KEY:
    print("GEMINI models that can analyse images:")
    for m in gclient.models.list():
        name = m.name.replace("models/", "")
        if "gemini" in name and "generateContent" in (m.supported_actions or []):
            print("  ", name)

if OPENAI_KEY:
    print("\nOPENAI chat models:")
    skip = ("audio", "realtime", "tts", "transcribe", "image", "search", "embedding", "moderation", "instruct")
    for mid in sorted(m.id for m in oclient.models.list()):
        if mid.startswith("gpt") and not any(s in mid for s in skip):
            print("  ", mid)

GEMINI models that can analyse images:
   gemini-2.5-flash
   gemini-2.5-pro
   gemini-2.5-flash-preview-tts
   gemini-2.5-pro-preview-tts
   gemini-flash-latest
   gemini-flash-lite-latest
   gemini-pro-latest
   gemini-2.5-flash-lite
   gemini-2.5-flash-image
   gemini-3-flash-preview
   gemini-3.1-pro-preview
   gemini-3.1-pro-preview-customtools
   gemini-3.1-flash-lite-preview
   gemini-3.1-flash-lite
   gemini-3-pro-image-preview
   gemini-3-pro-image
   gemini-3.1-flash-image-preview
   gemini-3.1-flash-image
   gemini-3.1-flash-lite-image
   gemini-nano-banana-2.1
   gemini-3.5-flash
   gemini-3.5-flash-lite
   gemini-omni-flash-preview
   gemini-omni-1.1-flash
   gemini-3.5-transcribe
   gemini-3.6-flash
   gemini-3.7-flash
   gemini-3.8-flash
   gemini-4-argon
   gemini-3.1-flash-tts-preview
   gemini-3.8-flash-tts
   gemini-3.8-flash-lite-tts
   gemini-robotics-er-2-preview
   gemini-2.5-computer-use-preview-10-2025

OPENAI chat models:
   gpt-3.5-turbo
   gpt-3.5-turbo-0125

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash-lite"     # <<< paste a model id from the list above, e.g. a current "gemini-...-flash" model
OPENAI_MODEL = " gpt-5.4-mini"     # <<< paste a model id from the list above, e.g. a current small "gpt-..." model
N_VAL = 100           # validation images per model and prompt (cost/time control)
SLEEP_S = {"gemini": 4.5, "openai": 0.5}   # pause between calls (Gemini free tier allows only a few requests per minute)

In [ ]:
DEFS = ("Classify the DIRECT DRIVER of a forest-loss event shown at the CENTRE of this Landsat 8 satellite image "
        "(about 5 km x 5 km, Indonesia, colours enhanced for visibility). Choose exactly one class:\n"
        "- Plantation: large-scale industrial plantations (oil palm, timber/pulpwood, other large plantations); regular "
        "geometric blocks and straight road grids.\n"
        "- Smallholder agriculture: small-scale farming, small-scale mixed or oil-palm plantations; small irregular fields.\n"
        "- Grassland shrubland: grassland or shrubs replacing forest, often after fire; no clear crops or structures.\n"
        "- Other: mining, logging roads, fish ponds, secondary forest regrowth, or other causes.\n")
ASK = ('Answer ONLY with JSON: {"label": "<Plantation|Smallholder agriculture|Grassland shrubland|Other>", '
       '"confidence": <0-1>, "reason": "<max 20 words>"}')
ZERO_SHOT = DEFS + ASK
FEW_SHOT_INTRO = DEFS + "Here are four labelled example images, one per class:"

ZF = zipfile.ZipFile(BASE / "data" / "ForestNetDataset.zip")
PREFIX = next(n for n in ZF.namelist() if n.endswith("train.csv"))[:-len("train.csv")]
def png_bytes(example_path):
    img = np.asarray(Image.open(io.BytesIO(ZF.read(f"{PREFIX}{example_path}/images/visible/composite.png"))).convert("RGB"))
    lo, hi = np.percentile(img, [2, 98]); s = np.clip((img.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
    buf = io.BytesIO(); Image.fromarray((s * 255).astype(np.uint8)).save(buf, "PNG"); return buf.getvalue()

rng = np.random.default_rng(SEED)
# few-shot examples: one TRAINING image per class (fixed, random with seed)
SHOTS = [(c, man.example_path.values[rng.choice(TR_IDX[man.y.values[TR_IDX] == CLS2ID[c]])]) for c in CLASSES]
# stratified validation sample (same images for every model and prompt)
va_df = man.iloc[VA_IDX]
frac = min(1.0, (5 if QUICK else N_VAL) / len(va_df))
VAL_SAMPLE = pd.concat([g.sample(max(1, round(len(g) * frac)), random_state=SEED) for _, g in va_df.groupby("merged_label")])
print("Validation sample:", len(VAL_SAMPLE), VAL_SAMPLE.merged_label.value_counts().to_dict())
print("Few-shot examples (training set):", SHOTS)

def parse_label(text):
    try:
        j = json.loads(re.search(r"\{.*\}", text, re.S).group(0)); lab = str(j.get("label", "")).strip().lower()
        conf = float(j.get("confidence", np.nan)) if j.get("confidence") is not None else np.nan
    except Exception:
        lab, conf = text.lower(), np.nan
    for c in CLASSES:
        if lab.startswith(c.lower().split()[0]):
            return c, conf
    return None, conf

def call_gemini(query_png, few_shot):
    parts = []
    if few_shot:
        parts.append(FEW_SHOT_INTRO)
        for c, p in SHOTS:
            parts += [f"Example - class: {c}", types.Part.from_bytes(data=png_bytes(p), mime_type="image/png")]
        parts.append("Now classify this image. " + ASK)
    else:
        parts.append(ZERO_SHOT)
    parts.append(types.Part.from_bytes(data=query_png, mime_type="image/png"))
    r = gclient.models.generate_content(model=GEMINI_MODEL, contents=parts,
                                        config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"))
    u = getattr(r, "usage_metadata", None)
    return r.text or "", {"in": getattr(u, "prompt_token_count", None), "out": getattr(u, "candidates_token_count", None)}

def call_openai(query_png, few_shot):
    img = lambda b: {"type": "input_image", "image_url": "data:image/png;base64," + base64.b64encode(b).decode()}
    content = []
    if few_shot:
        content.append({"type": "input_text", "text": FEW_SHOT_INTRO})
        for c, p in SHOTS:
            content += [{"type": "input_text", "text": f"Example - class: {c}"}, img(png_bytes(p))]
        content.append({"type": "input_text", "text": "Now classify this image. " + ASK})
    else:
        content.append({"type": "input_text", "text": ZERO_SHOT})
    content.append(img(query_png))
    r = oclient.responses.create(model=OPENAI_MODEL, input=[{"role": "user", "content": content}])
    u = getattr(r, "usage", None)
    return r.output_text or "", {"in": getattr(u, "input_tokens", None), "out": getattr(u, "output_tokens", None)}

def run_vlm(name, fn, model_id, few_shot):
    tag = f"S6-{name.upper()}-{'FEW' if few_shot else 'ZERO'}"
    out_csv = OUT6 / f"{tag}.csv"
    done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
    rows = done.to_dict("records")
    todo = [p for p in VAL_SAMPLE.example_path if done.empty or p not in set(done.example_path)]
    print(f"{tag} ({model_id}): {len(todo)} images to query, {len(rows)} already done")
    for p in todo:
        q = png_bytes(p); text, usage, err = "", {}, ""
        for attempt in range(4):
            try:
                t0 = time.perf_counter(); text, usage = fn(q, few_shot); lat = (time.perf_counter() - t0) * 1000; break
            except Exception as e:
                err = repr(e)[:150]; lat = np.nan; time.sleep(10 * (attempt + 1))
        lab, conf = parse_label(text)
        rows.append({"example_path": p, "true": man.set_index("example_path").merged_label[p], "pred": lab, "confidence": conf,
                     "raw": text[:300], "error": err, "latency_ms": lat, "tokens_in": usage.get("in"), "tokens_out": usage.get("out")})
        pd.DataFrame(rows).to_csv(out_csv, index=False)          # saved after every call, so a disconnect loses nothing
        time.sleep(SLEEP_S[name])
    df = pd.DataFrame(rows)
    y = df.true.map(CLS2ID).values; yp = df.pred.map(CLS2ID).fillna(-1).astype(int).values
    m = evaluate(y, yp); m["invalid_rate"] = float((yp == -1).mean()); m["median_latency_ms"] = float(np.nanmedian(df.latency_ms))
    m["mean_tokens_in"] = float(pd.to_numeric(df.tokens_in, errors="coerce").mean()); m["n"] = len(df)
    log_run(tag, f"{name}:{model_id}", "few-shot prompt" if few_shot else "zero-shot prompt",
            {"model": model_id, "n_val": len(df), "temperature": 0 if name == "gemini" else "default", "image": "contrast-stretched PNG"},
            SEED, m, infer_ms=m["median_latency_ms"], note=f"invalid answers {m['invalid_rate']:.1%}")
    print(f"{tag}: macro-F1 {m['macro_f1']:.3f} | invalid {m['invalid_rate']:.1%}")
    return m

Validation sample: 101 {'Plantation': 46, 'Smallholder agriculture': 30, 'Other': 15, 'Grassland shrubland': 10}
Few-shot examples (training set): [('Grassland shrubland', 'examples/-1.0995709144052763_121.59326201056416'), ('Other', 'examples/0.7485605657863448_104.2170944110189'), ('Plantation', 'examples/0.5281756694866891_102.28426604764368'), ('Smallholder agriculture', 'examples/-3.5310749810514563_102.9329527761482')]


In [ ]:
GEMINI_MODEL = "gemini-3.5-flash-lite"
for t in ["S6-GEMINI-ZERO", "S6-GEMINI-FEW"]:
    f = OUT6 / f"{t}.csv"
    if f.exists(): f.unlink()
text, usage = call_gemini(png_bytes(VAL_SAMPLE.example_path.iloc[0]), few_shot=False)
print("ANSWER:", text)

ANSWER: {
  "label": "Plantation",
  "confidence": 0.95,
  "reason": "Large-scale industrial blocks with straight grids characteristic of commercial plantations."
}


In [ ]:
log = pd.read_csv(LOG_PATH)
log = log[~log.run_id.astype(str).str.startswith(("S6-GEMINI", "S6-OPENAI"))]
log.to_csv(LOG_PATH, index=False); print("log cleaned")

log cleaned


In [ ]:
vlm_results = {}
if GEMINI_KEY and GEMINI_MODEL:
    for fs in (False, True):
        vlm_results[f"Gemini ({GEMINI_MODEL}) {'few-shot' if fs else 'zero-shot'}"] = run_vlm("gemini", call_gemini, GEMINI_MODEL, fs)
else:
    print("Gemini skipped (no key or GEMINI_MODEL not set).")

S6-GEMINI-ZERO (gemini-3.5-flash-lite): 0 images to query, 101 already done
S6-GEMINI-ZERO: macro-F1 0.417 | invalid 0.0%
S6-GEMINI-FEW (gemini-3.5-flash-lite): 3 images to query, 98 already done
S6-GEMINI-FEW: macro-F1 0.462 | invalid 0.0%


In [ ]:
   #not executing: Due to the api key issue
   for t in ["S6-OPENAI-ZERO", "S6-OPENAI-FEW"]:
       f = OUT6 / f"{t}.csv"
       if f.exists(): f.unlink(); print("deleted", f.name)
   log = pd.read_csv(LOG_PATH)
   log = log[~log.run_id.astype(str).str.startswith("S6-OPENAI")]
   log.to_csv(LOG_PATH, index=False)

In [ ]:
   #not executing: Due to the api key issue
   text, usage = call_openai(png_bytes(VAL_SAMPLE.example_path.iloc[0]), few_shot=False)
   print(text); print(usage)

NotFoundError: Error code: 404 - {'error': {'message': 'The model ` gpt-5.4-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [ ]:
   #not executing: Due to the api key issue
if OPENAI_KEY and OPENAI_MODEL:
    for fs in (False, True):
        vlm_results[f"GPT ({OPENAI_MODEL}) {'few-shot' if fs else 'zero-shot'}"] = run_vlm("openai", call_openai, OPENAI_MODEL, fs)
else:
    print("OpenAI skipped (no key or OPENAI_MODEL not set).")

S6-OPENAI-ZERO ( gpt-5.4-mini): 101 images to query, 0 already done


KeyboardInterrupt: 

In [ ]:
# Save the chosen prompt per model (best validation macro-F1) for the single test run in Stage 7
chosen = {}
for name, key in (("gemini", "Gemini"), ("openai", "GPT")):
    cands = {k: v for k, v in vlm_results.items() if k.startswith(key)}
    if cands:
        best = max(cands, key=lambda k: cands[k]["macro_f1"])
        chosen[name] = {"model": GEMINI_MODEL if name == "gemini" else OPENAI_MODEL, "few_shot": "few-shot" in best,
                        "val_macro_f1": cands[best]["macro_f1"]}
json.dump({"chosen": chosen, "zero_shot_prompt": ZERO_SHOT, "few_shot_intro": FEW_SHOT_INTRO, "ask": ASK,
           "few_shot_examples": SHOTS, "val_sample": list(VAL_SAMPLE.example_path), "image_preprocessing": "2-98 percentile stretch, PNG"},
          open(OUT6 / "vlm_prompts_final.json", "w"), indent=2)
print("Chosen prompts:", chosen)

Chosen prompts: {'gemini': {'model': 'gemini-3.5-flash-lite', 'few_shot': True, 'val_macro_f1': 0.46184640522875814}}


## Comparison (validation)


In [ ]:
s3 = BASE / "outputs" / "stage3" / "runs"
vit = [json.load(open(s3 / f"S3-VIT-FINAL-s{s}" / "result.json"))["val"] for s in FINAL_SEEDS if (s3 / f"S3-VIT-FINAL-s{s}" / "result.json").exists()]
rows = []
if vit:
    v = pd.DataFrame(vit); rows.append({"model": "ViT-Small (ours), full val, 3 seeds", "macro_f1": f"{v.macro_f1.mean():.3f} ± {v.macro_f1.std():.3f}",
                                         "recall_other": round(v.recall_other.mean(), 3), "rate_P_to_S": round(v.rate_P_to_S.mean(), 3)})
    pos = [int(np.where(man.example_path.values[VA_IDX] == p)[0][0]) for p in VAL_SAMPLE.example_path]
    vm = [evaluate(man.y.values[VA_IDX][pos], np.load(s3 / f"S3-VIT-FINAL-s{s}" / "val_pred.npy")[pos]) for s in FINAL_SEEDS
          if (s3 / f"S3-VIT-FINAL-s{s}" / "val_pred.npy").exists()]
    if vm:
        vmd = pd.DataFrame(vm); rows.append({"model": f"ViT-Small (ours), same {len(pos)}-image sample", "macro_f1": f"{vmd.macro_f1.mean():.3f} ± {vmd.macro_f1.std():.3f}",
                                              "recall_other": round(vmd.recall_other.mean(), 3), "rate_P_to_S": round(vmd.rate_P_to_S.mean(), 3)})
if tm_search:
    rows.append({"model": "TerraMind-1.0-base fine-tuned, full val, 3 seeds", "macro_f1": f"{tm_df.macro_f1.mean():.3f} ± {tm_df.macro_f1.std():.3f}",
                 "recall_other": round(tm_df.recall_other.mean(), 3), "rate_P_to_S": round(tm_df.rate_P_to_S.mean(), 3),
                 "train_min": round(tm_df.train_min.mean(), 1), "latency_ms": round(TM_LAT, 1)})
for k, m in vlm_results.items():
    rows.append({"model": k + f", {m['n']}-image sample", "macro_f1": f"{m['macro_f1']:.3f}", "recall_other": round(m["recall_other"], 3),
                 "rate_P_to_S": round(m["rate_P_to_S"], 3), "invalid_rate": round(m["invalid_rate"], 3),
                 "latency_ms": round(m["median_latency_ms"]), "tokens_in": round(m["mean_tokens_in"]) if not np.isnan(m["mean_tokens_in"]) else None})
comp6 = pd.DataFrame(rows); display(comp6); comp6.to_csv(OUT6 / "comparison_stage6.csv", index=False)

,model,macro_f1,recall_other,rate_P_to_S,train_min,latency_ms,invalid_rate,tokens_in
0,"ViT-Small (ours), full val, 3 seeds",0.760 ± 0.017,0.681,0.098,NaN,NaN,NaN,NaN
1,"ViT-Small (ours), same 101-image sample",0.799 ± 0.040,0.778,0.130,NaN,NaN,NaN,NaN
2,"TerraMind-1.0-base fine-tuned, full val, 3 seeds",0.743 ± 0.021,0.752,0.104,4.8,15.5,NaN,NaN
3,"Gemini (gemini-3.5-flash-lite) zero-shot, 101-image sample",0.417,0.800,0.022,NaN,2667.0,0.0,1278.0
4,"Gemini (gemini-3.5-flash-lite) few-shot, 101-image sample",0.462,0.867,0.000,NaN,2268.0,0.0,5675.0


In [ ]:
# Confusion matrices of the VLM runs (where do they go wrong?)
for k in vlm_results:
    tag = "S6-" + ("GEMINI" if k.startswith("Gemini") else "OPENAI") + ("-FEW" if "few-shot" in k else "-ZERO")
    df = pd.read_csv(OUT6 / f"{tag}.csv"); df["pred"] = df.pred.fillna("INVALID")
    print("\n", k); display(pd.crosstab(df.true, df.pred))


 Gemini (gemini-3.5-flash-lite) zero-shot


pred,Other,Plantation,Smallholder agriculture
true,,,
Grassland shrubland,1,8,1
Other,12,1,2
Plantation,10,35,1
Smallholder agriculture,7,13,10



 Gemini (gemini-3.5-flash-lite) few-shot


pred,Grassland shrubland,Other,Plantation,Smallholder agriculture
true,,,,
Grassland shrubland,1,2,7,0
Other,1,13,0,1
Plantation,1,10,35,0
Smallholder agriculture,2,5,14,9
